In [ ]:
import pandas as pd
import os
from datetime import datetime

def scan_and_process_orders():
    path = '/content/'
    files = [f for f in os.listdir(path) if f.endswith('.csv')]

    # Schema mục tiêu cho ORDERS
    target_cols = ['order_id', 'customer_id_FK', 'zip_FK', 'sales_employee_id_FK', 'order_date', 'order_status', 'device_type', 'order_source', 'comment']

    # Mapping dự đoán các tên cột tương đồng cho ORDERS
    potential_mappings = {
        'order_id': ['order_id', 'id', 'transaction_id'],
        'customer_id_FK': ['customer_id', 'cust_id', 'user_id', 'customer_id_fk'],
        'zip_FK': ['zip', 'zip_code', 'postal_code', 'zip_fk', 'zip_FK'],
        'sales_employee_id_FK': ['sales_employee_id', 'employee_id', 'staff_id', 'sales_rep_id'],
        'order_date': ['order_date', 'timestamp', 'created_at', 'date'],
        'order_status': ['order_status', 'status', 'delivery_status'],
        'device_type': ['device_type', 'device', 'platform', 'browser'],
        'order_source': ['order_source', 'source', 'channel', 'medium'],
        'comment': ['comment', 'notes', 'description', 'remarks']
    }

    source_report = {}
    collected_dfs = []

    print("--- Bắt đầu quét các file cho bảng ORDERS ---")
    for file in files:
        if file == 'orders_new.csv': continue
        try:
            file_path = os.path.join(path, file)
            # Sử dụng utf-8-sig để đọc đúng tiếng Việt
            temp_df = pd.read_csv(file_path, nrows=0, encoding='utf-8-sig')
            found_cols = {}

            for target, aliases in potential_mappings.items():
                for alias in aliases:
                    if alias in temp_df.columns:
                        found_cols[alias] = target
                        break

            if len(found_cols) >= 3:
                print(f"Tìm thấy dữ liệu ORDERS trong: {file} ({list(found_cols.keys())})")
                # Đọc toàn bộ file với encoding utf-8-sig
                full_df = pd.read_csv(file_path, encoding='utf-8-sig')
                mapped_df = full_df[list(found_cols.keys())].rename(columns=found_cols)
                collected_dfs.append(mapped_df)

                for alias, target in found_cols.items():
                    if target not in source_report:
                        source_report[target] = []
                    source_report[target].append(f"{file} (gốc: {alias})")
        except Exception:
            continue

    if not collected_dfs:
        print("Không tìm thấy file nào chứa dữ liệu orders.")
        return

    final_df = pd.concat(collected_dfs, ignore_index=True)
    initial_len = len(final_df)
    final_df = final_df.drop_duplicates()
    print(f"\nĐã xử lý: Xóa {initial_len - len(final_df)} dòng trùng lặp.")

    for col in target_cols:
        if col not in final_df.columns:
            final_df[col] = None

    defaults = {
        'order_id': 'UNKNOWN_ORD', 'customer_id_FK': 'UNKNOWN_CUST', 'zip_FK': '00000',
        'sales_employee_id_FK': 'N/A', 'order_date': datetime.now(),
        'order_status': 'Unknown', 'device_type': 'Unknown', 'order_source': 'Unknown', 'comment': ''
    }

    for col, val in defaults.items():
        final_df[col] = final_df[col].fillna(val)

    final_df['order_date'] = pd.to_datetime(final_df['order_date'], errors='coerce').fillna(datetime.now())

    # Lưu kết quả
    output_file = '/content/orders_new.csv'
    final_df[target_cols].to_csv(output_file, index=False, encoding='utf-8-sig')

    print(f"\n--- HOÀN THÀNH ---")
    print(f"File lưu tại: {output_file}")
    
    print("\n--- BÁO CÁO NGUỒN DỮ LIỆU (SOURCE REPORT) ---")
    for col in target_cols:
        sources = ", ".join(source_report.get(col, ["Không tìm thấy - Sử dụng mặc định"]))
        print(f"Cột '{col}': {sources}")

scan_and_process_orders()